    Работа с MySQL;
        Получение настроек торговых инструментов;
            Определение Минимального Количества Контрактов для объёма 0,01 lot;
            Создание sql таблицы с отредактированным Количеством Контрактов; 
            Обновление таблиц в sql базе.

In [1]:
# --- SETTINGS ---

# Скрипт работает в двух режимах: тестовом и реальном.
IS_TEST_MODE = True  # Смени на False для работы с реальными данными

# ====================== BOOTSTRAP ======================
import os
import sys
from pathlib import Path

file_dir = os.getcwd()                                                  # */[sub_project_dir]/ipynb_files/...
#sub_project_dir = Path(file_dir).parent
#project_dir = sub_project_dir.parent
parent_dir = Path(file_dir).parent.parent.parent   # Поднимемся на 3 уровня выше, чтобы попасть в корень проекта

libraries_path = str(parent_dir / "libraries_py")
if libraries_path not in sys.path: sys.path.insert(0, libraries_path)

# ====================== ОСНОВНОЙ КОД ======================
from project_config import ProjectConfig

paths = ProjectConfig(file_dir=file_dir)

print(f"✅ Режим тестирования: {'ВКЛЮЧЕН' if IS_TEST_MODE else 'ВЫКЛЮЧЕН'}")
print(f"✅ Путь к файлу: {paths.input_samples_data}")
"""# Пример использования
print(paths.input_log_data)
print(paths.output_log_data)
print(paths.project_dir)"""

Заведомо существующие директории:
    📁 [file_dir]; Путь к директории с ipynb/py файлами: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\calculation_min_symbol_number_contracts\ipynb_files
    📁 [sub_project_dir]; Путь к директории СубПроекта: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\calculation_min_symbol_number_contracts
    📁 [project_dir]; Путь к директории Проекта: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
    📁 [parent_dir]; Путь к директории для доступа к библиотекам: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations

Директории СубПроекта:
  Директории Исходных Данных:
    📁 [input_log_data]; Путь к каталогу с логами исходных файлов: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\calculation_min_symbol_number_contracts\input_data\input_log_data
    📁 [input_temp_data]; Путь к каталогу с временными файлами: c:\unique_data\rep_fo_metatrader_server\fc_t

'# Пример использования\nprint(paths.input_log_data)\nprint(paths.output_log_data)\nprint(paths.project_dir)'

In [2]:
# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                    # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                               # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                           # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                        # Вывод ДФ
                "df_to_csv",                            # Сохранение ДФ в CSV 
                "CSVLoader",
                "save_data_log_work_file",
                "detect_encoding",
                "time_to_minutes",
                "load_string_list",
                "save_dict_log_work_file",
                "load_csv",
                "list_print",
                ],
    "sql_request_2":
        [libraries_path, 
                "pd_read_sql",
                "load_mysql_tab"]
                }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

import pandas as pd


 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['pd_set_option', 'df_to_csv', 'CSVLoader', 'save_data_log_work_file', 'detect_encoding', 'time_to_minutes', 'load_string_list', 'save_dict_log_work_file', 'load_csv', 'list_print']
Импорт из 'sql_request_2' успешен: ['pd_read_sql', 'load_mysql_tab']

 Импортированные функции и их параметры:
Функция 'pd_set_option' из модуля 'yar_sed_general_lib' ожидает параметры: name_df: str, df: pandas.core.frame.DataFrame, rows: int = 10, columns: int | None = None, min_rows: int | None = None, width: int = 100
Функция 'df_to_csv' из модуля 'yar_sed_general_lib' ожидает параметры: df, csv_file_path
Функция 'CSVLoader' из модуля 'yar_sed_general_lib' ожидает параметры: file_path, delimiter=';', encoding='utf-8', df_name='dataframe'
Функция 'save_data_log_work_file' из модуля 'yar_sed_general_lib' ожидает параметры: df, file_name, directory_data_temp_files, directory_data_log_files
Функция 'detect_encoding' из

In [3]:
if not IS_TEST_MODE:
    # PROD version: Загрузка данных SQL таблицы с глобальными настройками символов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
    #cred_dotbig_file_relative_path  = "own_platform\\credits\\own_platforn_sql_main_main_01.txt"    # Поключение к внешней PROD базе
    cred_dotbig_file_relative_path  = "own_platform\\credits\\local_db.txt"                          # Подключение к локальной базе 
    #cred_fc_file_relative_path      = "credits\\sql.text"                                           #c:\\unique_data\\rep_fo_metatrader_server

    dotbig_quotes_df =  imported["load_mysql_tab"]("symbolsQuotes", cred_dotbig_file_relative_path)
    dotbig_symbols_df = imported["load_mysql_tab"]("symbols", cred_dotbig_file_relative_path)

    imported["save_data_log_work_file"](dotbig_quotes_df, "dotbig_quotes_df.csv", paths.input_temp_data, paths.input_log_data)
    imported["save_data_log_work_file"](dotbig_symbols_df, "dotbig_symbols_df.csv", paths.input_temp_data, paths.input_log_data)

In [6]:
if IS_TEST_MODE:    
    print("⚠️ Attention: Загружаеем ТЕСТОВЫЕ данные из CSV файла для дальнейшей работы")
    dotbig_quotes_df  = imported["load_csv"]("dotbig_quotes_df.csv",  paths.input_samples_data)
    dotbig_symbols_df = imported["load_csv"]("dotbig_symbols_df.csv", paths.input_samples_data)

⚠️ Attention: Загружаеем ТЕСТОВЫЕ данные из CSV файла для дальнейшей работы
✅ Success: [class CSVLoader]: DataFrame 'my_dataframe' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\calculation_min_symbol_number_contracts\input_data\input_samples\dotbig_quotes_df.csv'.
✅ Success: [class CSVLoader]: DataFrame 'my_dataframe' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\calculation_min_symbol_number_contracts\input_data\input_samples\dotbig_symbols_df.csv'.


In [8]:
# [ НЕ ОБЯЗАТЕЛЕН ] Выводим на экран загруженные DataFrame'ы и объединяем их в один DataFrame <<<<<<<<<<<<<<
imported["pd_set_option"]("\n dotbig_quotes_df",   dotbig_quotes_df,   3)
imported["pd_set_option"]("\n dotbig_symbols_df",  dotbig_symbols_df,  3)



 dotbig_quotes_df  (4,951 строк × 9 колонок)


,symbolId,ask,bid,middleQuote,spread,upDown,createdTimestamp,createdTimestampMs,updatedAt
0,1,0.98424,0.98225,0.983245,0.00199,1,1784152132,1784152132799,2026-07-15 21:48:56
...,...,...,...,...,...,...,...,...,...
4950,10830,50.60000,50.44000,50.520000,0.16000,1,1784145548,1784145548188,2026-07-15 19:59:08




 dotbig_symbols_df  (4,683 строк × 31 колонок)


,symbolId,name,hedgedMarginPercent,isGap,gapThresholdPercent,gapConfirmTicksCount,displayName,path,marketId,marginInitialBuy,marginInitialSell,precision,spread,source,baseCurrency,profitCurrency,tradeMode,calcMode,contractSize,volumeMax,volumeMin,volumeStep,volumeDefault,swapMode,swap3days,swapLong,swapShort,swapProfileId,markupEnabled,maxQuoteDelay,popularity
0,1,AUDCAD,0.0,0,0.5,3,AUD / CAD,FOREXFx Cross RatesAUDCAD,2,0.0025,0.0025,5,0.0,NaN,AUD,CAD,4,0,100000.0,1000.0,0.01,0.01,1.0,6.0,3.0,0.0,0.0,NaN,1,420,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4682,10830,NVO.N,0.0,0,0.5,3,Novo Nordisk,STOCKSCFDs - Stocks USNVO.N,33,0.1000,0.1000,2,0.0,NaN,USD,USD,4,2,100.0,1000.0,0.01,0.01,1.0,6.0,3.0,0.0,0.0,NaN,0,0,0.0


In [9]:
# Объединяем DataFrame'ы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
dotbig_symbols_quotes_df = pd.merge(dotbig_quotes_df, dotbig_symbols_df[['precision', 'symbolId', 'name', 'path']],on='symbolId',how='left')
imported["pd_set_option"]("\n dotbig_symbols_quotes_df",       dotbig_symbols_quotes_df,       3)



 dotbig_symbols_quotes_df  (4,951 строк × 12 колонок)


,symbolId,ask,bid,middleQuote,spread,upDown,createdTimestamp,createdTimestampMs,updatedAt,precision,name,path
0,1,0.98424,0.98225,0.983245,0.00199,1,1784152132,1784152132799,2026-07-15 21:48:56,5.0,AUDCAD,FOREXFx Cross RatesAUDCAD
...,...,...,...,...,...,...,...,...,...,...,...,...
4950,10830,50.60000,50.44000,50.520000,0.16000,1,1784145548,1784145548188,2026-07-15 19:59:08,2.0,NVO.N,STOCKSCFDs - Stocks USNVO.N


In [10]:
# Загружаем список криптовалют из файла crypto_list.txt, сформированного в МТ5client <<<<<<<<<<<<<<<<<<<<<<
if not IS_TEST_MODE:
    file_path = Path(os.environ["APPDATA"], "MetaQuotes", "Terminal", "Common", "Files", "crypto_list.txt")                      # Чтение файла и создание списка
else:
    print("⚠️ Attention: Загружаеем ТЕСТОВЫЕ данные из CSV файла для дальнейшей работы")
    file_path = Path(paths.input_samples_data, "crypto_list.txt")
    
with open(file_path, 'r') as file: crypto_list = [line.strip() for line in file]                        # Чтение файла и создание списка
imported["list_print"](crypto_list, "crypto_list",  5)

# отфильтровать строки в которых столбец 'name' содержит любое из значений в списке crypto_list
is_crypto = dotbig_symbols_quotes_df['name'].str.contains('|'.join(crypto_list), case=False, na=False)
crypto_symbols_df = dotbig_symbols_quotes_df[is_crypto]
imported["pd_set_option"]("\n crypto_symbols_df",       crypto_symbols_df,       5)

⚠️ Attention: Загружаеем ТЕСТОВЫЕ данные из CSV файла для дальнейшей работы
📝 [ 995 ] элементов в списке [ crypto_list ] список: ['ADAUSD', 'APEUSD', 'ATMUSD', 'BATUSD', 'BCHUSD']...


 crypto_symbols_df  (730 строк × 12 колонок)


,symbolId,ask,bid,middleQuote,spread,upDown,createdTimestamp,createdTimestampMs,updatedAt,precision,name,path
371,372,0.07640,0.07250,0.074450,0.00390,1,1784152124,1784152124075,2026-07-15 21:48:44,4.0,1INCHUSD,CRYPTOCrypto1INCHUSD
...,...,...,...,...,...,...,...,...,...,...,...,...
3857,9364,0.32612,0.32273,0.324425,0.00339,1,1784152137,1784152137179,2026-07-15 21:48:59,5.0,TRXUSD,CRYPTOCrypto STDTRXUSD


In [ ]:
# Не Обязателен к выполнению <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
filtered_df = crypto_symbols_df[crypto_symbols_df['name'] == 'XBTJPY']
imported["pd_set_option"]("\n fitred_df",       filtered_df,       30)

In [ ]:
# Pасчёт минимального размера контракта для торговли 0,01 лота, что бы стоимость пункта была отображаема <<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
cols = ['spread', 'precision']
crypto_symbols_df[cols] = crypto_symbols_df[cols].apply(pd.to_numeric, errors='coerce')

# Создаем пустой DataFrame для хранения результатов
symbols_contracts_df = pd.DataFrame(columns=['Symbol', 'spread', 'new_contract_size_fo_iter', 
                                             'min_lot_pips_cost', 'new_min_lot_pips_cost_fo_iter', 
                                             'min_lot_spread_cost', 'new_spread_cost_fo_iter'])

Lot = 0.01  

# Перебираем строки DataFrame, где 'Path' начинается с path_startswith
for index, row in crypto_symbols_df.iterrows():
        
    # Инициализация переменных для итерации
    new_contract_size_fo_iter = 1  # Начальный размер контракта
    new_spread_cost_fo_iter = 0
    new_min_lot_pips_cost_fo_iter = 0

    spread = row["spread"]
    symbol = row["name"]
    point = row["precision"]
    if point < 1: point = 0.1 # Защита от деления на ноль
    """print(f"\nProcessing symbol: {symbol}, spread: {spread}, precision (point): {point}")"""

    if pd.isna(spread) or spread <= 0: spread = 1/(10**point) # Если спред не задан, устанавливаем минимально возможный
    print(f"Initial spread for {symbol} set to {spread}")    
    """
    print(f"symbol = {symbol}, new_contract_size = {new_contract_size_fo_iter}, "
          f"new_spread_cost_fo_iter = {new_spread_cost_fo_iter}, "
          f"new_min_lot_pips_cost_fo_iter = {new_min_lot_pips_cost_fo_iter}")"""


    # Итерация для определения минимального размера контракта <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
    while new_spread_cost_fo_iter < 0.1 or new_min_lot_pips_cost_fo_iter < 0.0001:
        new_spread_cost_fo_iter = round(spread * new_contract_size_fo_iter * Lot, 2)
        #print(f"point = {point}, new_contract_size_fo_iter = {new_contract_size_fo_iter}, Lot = {Lot}")
        new_min_lot_pips_cost_fo_iter = 1/(10**point) * new_contract_size_fo_iter * Lot
        #print(f"new_contract_size = {new_contract_size_fo_iter}, new_spread_cost = {new_spread_cost_fo_iter}, new_min_lot_pips_cost_fo_iter = {new_min_lot_pips_cost_fo_iter}")
        new_contract_size_fo_iter *= 10
    
    # Сохраняем значения без последнего увеличения
    """new_spread_cost_fo_iter = round(spread * new_contract_size_fo_iter * Lot, 2)
    new_min_lot_pips_cost_fo_iter = point * new_contract_size_fo_iter * Lot"""
    
    new_contract_size_fo_iter /= 10
    """print(f"a_symbol = {symbol}, new_contract_size = {new_contract_size_fo_iter}, "
          f"new_spread_cost_fo_iter = {new_spread_cost_fo_iter}")"""
    
    # Добавляем данные в DataFrame
    symbols_contracts_df.loc[len(symbols_contracts_df)] = {
        'Symbol': symbol,
        'spread': round(spread, 5),
        'new_contract_size_fo_iter': int(new_contract_size_fo_iter),
        #'min_lot_spread_cost': round(min_lot_spread_cost, 5),
        'new_spread_cost_fo_iter': round(new_spread_cost_fo_iter, 5),
        #'min_lot_pips_cost': round(min_lot_pips_cost, 5),
        'new_min_lot_pips_cost_fo_iter': round(new_min_lot_pips_cost_fo_iter, 5)
    }

symbols_contracts_df.drop(columns=["min_lot_pips_cost", "min_lot_spread_cost"], inplace=True)  

imported["pd_set_option"]("минимальный размер контракта, для 0,01 лота что бы стоимость пункта была отображаема", symbols_contracts_df, 5)

минимальный размер контракта, для 0,01 лота что бы стоимость пункта была отображаема  (730 строк × 5 колонок)

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Symbol</th>
      <th>spread</th>
      <th>new_contract_size_fo_iter</th>
      <th>new_min_lot_pips_cost_fo_iter</th>
      <th>new_spread_cost_fo_iter</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1INCHUSD</td>
      <td>0.00390</td>
      <td>10000</td>
      <td>0.010</td>
      <td>0.39</td>
    </tr>
    <tr>
      <th>...</th>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <th>729</th>
      <td>TRXUSD</td>
      <td>0.00339</td>
      <td>10000</td>
      <td>0.001</td>
      <td>0.34</td>
    </tr>
  </tbody>
</table>
<p>730 rows × 5 columns</p>
</div>

In [12]:
# Создаем словарь из symbols_contracts_df <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
symbols_contracts_dict = {row['Symbol']: {'contractSize_s': row['new_contract_size_fo_iter']}for index, row in symbols_contracts_df.iterrows()}
print(f"\n✅ Создан словарь symbols_contracts_dict с минимальными размерами контрактов для торговли 0,01 лота:\n{symbols_contracts_dict}")

# Сохраняем словарь в файл JSON и в лог-файл <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
imported["save_dict_log_work_file"](symbols_contracts_dict, "symbols_contracts_dict.json", paths.output_temp_data, paths.output_log_data)


✅ Создан словарь symbols_contracts_dict с минимальными размерами контрактов для торговли 0,01 лота:
{'1INCHUSD': {'contractSize_s': 10000}, 'AAVEUSD': {'contractSize_s': 100}, 'ALGUSD': {'contractSize_s': 10000}, 'ALICEUSD': {'contractSize_s': 1000}, 'ALPHAUSD': {'contractSize_s': 10000}, 'ANKRUSD': {'contractSize_s': 100000}, 'ANTUSD': {'contractSize_s': 100}, 'APE3USD': {'contractSize_s': 1000}, 'ARUSD': {'contractSize_s': 100}, 'ATOMUSD': {'contractSize_s': 1000}, 'AUDIOUSD': {'contractSize_s': 10000}, 'AVAEUR': {'contractSize_s': 1000}, 'AVAGBP': {'contractSize_s': 1000}, 'AVATRL': {'contractSize_s': 10}, 'AVAUSD': {'contractSize_s': 1000}, 'AVXUSD': {'contractSize_s': 1000}, 'AXSUSD': {'contractSize_s': 100}, 'BALUSD': {'contractSize_s': 100000}, 'BANDUSD': {'contractSize_s': 10000}, 'BSVUSD': {'contractSize_s': 100}, 'C98USD': {'contractSize_s': 10000}, 'CAKEUSD': {'contractSize_s': 100000}, 'CELOUSD': {'contractSize_s': 1000}, 'CELRUSD': {'contractSize_s': 100000}, 'CHRUSD': {'